# 🤝 21 · Learning to align without labels: Gum-Net and equivariant transformers (BOE-ViT)
### Spectral pooling, feature correlation, spatial transformers, missing-wedge imputation — and why architectures should respect geometry

**BiologyToResearch** · Part E — Unsupervised & geometric deep learning (2020–2026) · Module 21 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/21_deep_alignment.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU is enough (≈ 3–4 min); a T4 GPU makes training faster but is optional |
| **Core frameworks** | Python 3, PyTorch (`affine_grid`, `grid_sample`, `torch.fft`), NumPy, SciPy, Matplotlib |
| **Prerequisites** | Modules 03 (Fourier, DCT), 05 (PyTorch), 07 (missing wedge), 12 (constrained correlation, alignment), 16 (CNNs) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | **Zeng & Xu 2020** (CVPR — Gum-Net); **Jiang *et al.* 2025** (CVPR — BOE-ViT) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. The
> networks here are small **2D** versions ("Gum-Net-lite") of 3D methods, trained for about a minute on projections of
> code-generated molecules. They illustrate the core ideas; they do **not** reproduce the papers' architectures or
> numbers. **All molecules and data are generated by code** (nothing is downloaded): realistic stand-ins with the size
> and symmetry of the PDB entries they are named after.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Frame** subtomogram alignment as *regression* of transformation parameters by a network, trained **without** pose labels.
2. **Explain** and **demonstrate** why max pooling discards the exact positions alignment needs, and how DCT
   **spectral pooling** keeps them.
3. **Build** the four parts of Gum-Net — a Siamese feature extractor, a two-way **feature-correlation layer**, a
   parameter regressor and a **differentiable spatial transformer** — plus **missing-wedge imputation** and the
   unsupervised loss $1-r$.
4. **Train** a 2D Gum-Net-lite, **compare** it with exhaustive search and with a max-pooling ablation, and use it for
   **reference-free averaging**.
5. **Define** equivariance, **show** that ordinary ViT patch embedding is not shift-equivariant, and **fix** it with the
   polyphase anchoring used by BOE-ViT.
6. **Train** a self-supervised pose regressor on self-generated transformations, as BOE-ViT does.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, a bank of molecule projections | 5 min |
| 1 · Alignment as regression | the idea; the four building blocks one by one | 35 min |
| 2 · Gum-Net-lite | assemble, train without labels, evaluate, ablate | 35 min |
| 3 · Reference-free averaging | iterate "align to the average, re-average" | 15 min |
| 4 · Equivariance and BOE-ViT | patch embeddings, polyphase anchoring, rotation encodings | 30 min |
| 5 · Self-supervised pose regression | BOE-ViT's training signal in miniature | 15 min |
| 📄 / ✅ / 🧠 / 🚀 | paper connections, checkpoint, discussion, going further | 15 min |

## 🧫 The biological question

Subtomogram averaging turns thousands of noisy copies of a molecule into one clear structure — but only after every copy
has been rotated and shifted onto a common frame. In Modules 12–13 each alignment was a *search*: score thousands of
rotations, then refine. For a dataset of 10⁵ particles aligned to several references over many rounds, that is weeks of
CPU time. In 2020, Xiangrui Zeng and Min Xu asked: can a neural network **look at two subtomograms and directly output
the transformation between them** — in milliseconds — and learn to do so **without anyone telling it the true poses**?
Their answer, Gum-Net, was 70–110 times faster than classical methods and more accurate at low SNR. Five years later,
BOE-ViT asked how to make transformer networks respect the geometry of shifts and rotations.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — fine, everything runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays) of the code-generated stand-in for zoo entry `pid` (Å, centred at the origin)."""
    pid = pid.lower()
    if pid in _ATOMS_CACHE:
        return _ATOMS_CACHE[pid]
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    _ATOMS_CACHE[pid] = atoms
    return atoms


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of the code-generated stand-in for a zoo entry (no download)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, f"code-generated stand-in modelled on PDB {pid.upper()}"


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
print("🧰 toolkit loaded")

### 0.1 The data: a bank of 2D views of asymmetric molecules

Gum-Net works on 3D subtomograms. To train in a minute on a CPU, we use the **2D analogue**: images (projections) of
molecules, which differ by an in-plane rotation $\theta$ and a shift $(t_x,t_y)$ — 3 parameters instead of 6. We use only
**asymmetric** molecules: a top view of the 20S proteasome is a ring, whose rotation angle is undefined, which would
make "the correct answer" meaningless.

In [ ]:
# ── 0.1 Projection bank: random views of five asymmetric complexes ──
S = 32                                                        # image size (pixels; 1 pixel = 1 nm)
CAST = ["4v4r", "1qvr", "4v7r", "5mrc", "4cr2"]               # 70S & 80S ribosomes, ClpB, mitoribosome, 26S proteasome
mols = []
for pid in CAST:
    vol, vs, src = zoo_map(pid)
    mols.append(prepare_map(vol, size=S, sigma=1.0))
    print(f"{pid.upper()} {ZOO[pid]['name']:24s} ← {src}")


def projection_bank(n, rng):
    """n normalised projections of randomly chosen molecules in random 3D orientations (shape n×S×S)."""
    out = [normalize(rotate_volume(mols[rng.integers(len(mols))], random_rotation(rng)).sum(axis=0)) for _ in range(n)]
    return torch.tensor(np.stack(out))


t0 = time.time()
bank, test_bank = projection_bank(600, rng), projection_bank(200, rng)   # training views / never-seen test views
print(f"{len(bank)} training and {len(test_bank)} test views in {time.time() - t0:.1f} s")
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, im in zip(axes, bank[:8]):
    ax.imshow(im); ax.axis("off")
plt.suptitle("a few training views (different molecules, different orientations)", fontsize=9); plt.show()

---
## 1 · Alignment as regression — and its four building blocks

📍 **Where we are.** In Module 12 we aligned by *searching*: score many candidate rotations with constrained correlation
and keep the best. Here a network *predicts* the transformation in one forward pass. The challenge is training it: we
have no true poses for real data. Gum-Net's trick is to make the whole pipeline differentiable, so that the network can
learn simply by **making the transformed subtomogram look like its partner**.

```text
  s_a ─► feature ─┐                                   ┌──────────── spatial transformer ◄── s_b
                  ├─► correlation (a→b, b→a) ─► regressor ─► (θ, t) ──┘        │
  s_b ─► feature ─┘    (shared weights)                               missing-wedge imputation
                                                                               │
                                         loss = 1 − Pearson(s_a, transformed s_b)
```

### 1.1 The pairs: a subtomogram and a transformed, wedge-limited partner

Like the paper's training data, the first image $s_a$ is complete and the second $s_b$ is missing a wedge of Fourier
space (in 2D: two opposite sectors). We generate $s_b$ by rotating and shifting $s_a$'s underlying view by a random
$(\theta,t)$, adding independent noise, and removing the wedge. *(The paper trains on random pairs of subtomograms
drawn from the dataset; making $s_b$ from the same view keeps our 2D toy simple.)*

In [ ]:
# ── 1.1 Differentiable 2D transforms, 2D wedge masks and training pairs ──
MAX_ANGLE, MAX_SHIFT = math.radians(45), 4.0                 # range of the random transformations
OUT_SHIFT = MAX_SHIFT * math.sqrt(2)                         # undoing a rotated shift can need up to √2 × more
FY, FX = torch.fft.fftfreq(S)[:, None], torch.fft.fftfreq(S)[None, :]


def rot_matrices(theta):
    """Batch of 2×2 rotation matrices acting on (x, y)."""
    c, s = torch.cos(theta), torch.sin(theta)
    return torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2)


def transform(x, theta, t):
    """Spatial transformer: rotate each image (B×1×S×S) by θ about its centre, then shift by t (pixels, (x, y))."""
    Rt = rot_matrices(theta).transpose(1, 2)                  # output → input coordinates need the inverse rotation
    tn = t * 2 / (S - 1)                                      # pixels → normalised [-1, 1] units
    theta_aff = torch.cat([Rt, -Rt @ tn[:, :, None]], 2)
    grid = F.affine_grid(theta_aff, list(x.shape), align_corners=True)
    return F.grid_sample(x, grid, align_corners=True)


def wedge2d(theta, tilt_deg=60.0):
    """2D 'missing wedge' masks of images rotated by θ: M(Rᵀk) with M(k) = 1(|k_y| ≤ |k_x| tan(tilt))."""
    c, s = torch.cos(theta)[:, None, None], torch.sin(theta)[:, None, None]
    kx, ky = c * FX + s * FY, -s * FX + c * FY
    m = (ky.abs() <= kx.abs() * math.tan(math.radians(tilt_deg)) + 1e-9).float()
    m[:, 0, 0] = 1
    return m[:, None]


def make_pairs(src, n, snr, generator=None):
    """(s_a, s_b, θ_true, t_true): s_b = wedge(T_θ,t(view) + noise); s_a = view + noise (no wedge)."""
    idx = torch.randint(len(src), (n,), generator=generator)
    a = src[idx][:, None]
    th = (torch.rand(n, generator=generator) * 2 - 1) * MAX_ANGLE
    t = (torch.rand(n, 2, generator=generator) * 2 - 1) * MAX_SHIFT
    b = transform(a, th, t)
    sa = a + torch.randn(a.shape, generator=generator) / math.sqrt(snr)
    sb = b + torch.randn(b.shape, generator=generator) / math.sqrt(snr)
    sb = torch.fft.ifft2(torch.fft.fft2(sb) * wedge2d(torch.zeros(n))).real
    return sa, sb, th, t


def undo_params(th, t):
    """The transformation that maps s_b back onto s_a: rotate by −θ, then shift by −R(−θ)·t."""
    return -th, -(rot_matrices(-th) @ t[:, :, None])[:, :, 0]


sa, sb, th, t = make_pairs(test_bank, 4, snr=100.0, generator=torch.Generator().manual_seed(0))
back_th, back_t = undo_params(th, t)
restored = transform(sb, back_th, back_t)
fig, axes = plt.subplots(3, 4, figsize=(7.5, 5.6))
for j in range(4):
    for i, (im, lab) in enumerate([(sa, "s_a"), (sb, f"s_b (θ={math.degrees(th[j]):.0f}°)"), (restored, "s_b transformed back")]):
        axes[i, j].imshow(im[j, 0]); axes[i, j].set_title(lab, fontsize=8); axes[i, j].axis("off")
plt.tight_layout(); plt.show()

🔍 The bottom row applies the *known* inverse transformation with our differentiable **spatial transformer** —
`affine_grid` builds, for every output pixel, the location to read from, and `grid_sample` interpolates there. Because
both operations are differentiable, a loss computed on the transformed image can send gradients back into the predicted
angle and shift. The restored images match $s_a$ except inside the missing wedge — exactly the problem imputation solves
(1.4).

### 1.2 Pooling: why max pooling hurts alignment, and DCT spectral pooling

Classification networks shrink their feature maps with **max pooling** (keep the largest value in each 2×2 block). That
throws away *where inside the block* a feature was — irrelevant for "is it a ribosome?", fatal for "by how much is it
shifted?". Gum-Net instead uses **spectral pooling**: take the type-II discrete cosine transform (DCT; Module 03), keep
only the low-frequency block, and transform back on the smaller grid. In 3D the paper writes the DCT as

$$C(v)_{lhw}=\tfrac{8}{LWH}\,\varepsilon_l\varepsilon_h\varepsilon_w\sum_{i,j,k} v_{ijk}\cos\tfrac{l\pi(2i+1)}{2L}\cos\tfrac{h\pi(2j+1)}{2H}\cos\tfrac{w\pi(2k+1)}{2W},$$

with $\varepsilon_0 = 1/\sqrt2$ and $\varepsilon = 1$ otherwise. *In words:* express the feature map as a sum of smooth
cosine waves, keep the slow ones, and rebuild — a downsampled, low-pass-filtered map in which a sub-pixel shift of a
feature is still visible as a smooth change.

In [ ]:
# ── 1.2 Spectral pooling vs max pooling: is a sub-pixel shift still visible after pooling? ──
def dct_matrix(n):
    """Orthonormal DCT-II matrix D (n×n): y = D x, x = Dᵀ y."""
    k, i = torch.arange(n)[:, None].float(), torch.arange(n)[None].float()
    D = torch.cos(math.pi * k * (2 * i + 1) / (2 * n)) * math.sqrt(2 / n)
    D[0] /= math.sqrt(2)
    return D


class SpectralPool2d(nn.Module):
    """Downsample n×n → m×m by keeping the m×m lowest DCT coefficients (Gum-Net's spectral pooling, in 2D)."""

    def __init__(self, n, m):
        super().__init__()
        self.register_buffer("Dn", dct_matrix(n))
        self.register_buffer("Dm", dct_matrix(m))
        self.m = m

    def forward(self, x):
        u = self.Dn @ x @ self.Dn.T                           # 2D DCT of every feature map
        u = u[..., :self.m, :self.m] * (self.m / x.shape[-1])  # crop the low-frequency block (keep the mean level)
        return self.Dm.T @ u @ self.Dm                        # inverse DCT on the smaller grid


def blob_at(x0, sigma=0.7):
    """A sharp Gaussian 'feature' (narrower than a 4-pixel pooling block) at horizontal position x0 on a 32×32 map."""
    yy, xx = torch.meshgrid(torch.arange(S).float(), torch.arange(S).float(), indexing="ij")
    return torch.exp(-((xx - x0) ** 2 + (yy - 15.5) ** 2) / (2 * sigma**2))[None, None]


def centroid_x(m):
    """Intensity-weighted mean x position of a (1×1×n×n) map, in pixels of the ORIGINAL 32-pixel grid."""
    m = m.clamp(min=0)[0, 0]; scale = S / m.shape[-1]
    xs = (torch.arange(m.shape[-1]).float() + 0.5) * scale - 0.5
    return float((m.sum(0) * xs).sum() / m.sum())


spool, mpool = SpectralPool2d(32, 8), nn.MaxPool2d(4)
true_x = np.linspace(14, 18, 21)
est = {"spectral pooling": [centroid_x(spool(blob_at(x))) for x in true_x],
       "max pooling": [centroid_x(mpool(blob_at(x))) for x in true_x]}
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(true_x, true_x, "k--", lw=1, label="perfect")
for lab, e in est.items():
    ax.plot(true_x, e, "o-", ms=3, label=lab)
ax.set_xlabel("true feature position (pixels)"); ax.set_ylabel("position read from 8×8 pooled map")
ax.set_title("After 4× downsampling, where is the feature?"); ax.legend(fontsize=8); plt.show()
for lab, e in est.items():
    print(f"{lab:17s}: mean position error {np.mean(np.abs(np.array(e) - true_x)):.3f} px")

🔍 For a sharp feature (narrower than the 4-pixel pooling block), max pooling reports nearly the same pooled map while
the feature moves inside a block — its position estimate lags in steps — whereas spectral pooling follows the true
position closely. *(For broad, smooth features the difference shrinks; try `sigma=1.5` in `blob_at`.)* Gum-Net's authors put it this way: "the exact feature spatial location is critical and the
information loss will lead to inaccurate downstream matching."

### 1.3 The feature-correlation layer

After the shared ("Siamese") feature extractor, every image is a grid of feature vectors $v(l,w)$, each scaled to unit
length. The **correlation layer** compares *every* position of $s_a$ with *every* position of $s_b$:

$$c_{ab}(l,w,n) = \big\langle v_a(l,w,:),\; v_b(\mathbf p_n,:)\big\rangle ,$$

where $\mathbf p_n$ runs over all positions of $s_b$. For an $8\times 8$ map this gives $64$ similarity maps — one per
position of $s_b$ — each saying "where in $s_a$ does this piece of $s_b$ appear?". Gum-Net computes **both directions**
($c_{ab}$ and $c_{ba}$, the "Siamese correlation layer"), whereas the 2D image-matching network it builds on (Rocco *et
al.*) used one. *In words:* the network does not have to discover matching by itself — it is handed a table of all
pairwise similarities, from which the transformation is easy to read.

In [ ]:
# ── 1.3 Correlation layer (both directions) ──
def correlation(fa, fb):
    """c[b, n, l, w] = <fa[b,:,l,w], fb[b,:,p_n]> for every position p_n of fb; then ReLU and L2-normalise over n."""
    B, C, H, W = fa.shape
    c = torch.einsum("bchw,bcn->bnhw", fa, fb.reshape(B, C, H * W))
    return F.normalize(F.relu(c), dim=1)


# demo with raw pixels as 'features': where does the piece of s_b at its centre appear in s_a?
fa_demo = F.normalize(F.avg_pool2d(sa[:1], 4).repeat(1, 4, 1, 1) + 0.01 * torch.randn(1, 4, 8, 8), dim=1)
fb_demo = F.normalize(F.avg_pool2d(sb[:1], 4).repeat(1, 4, 1, 1) + 0.01 * torch.randn(1, 4, 8, 8), dim=1)
c_demo = correlation(fa_demo, fb_demo)
print("correlation tensor shape (batch, positions of s_b, rows of s_a, columns of s_a):", tuple(c_demo.shape))

### 1.4 Missing-wedge imputation and the unsupervised loss

After the spatial transformer rotates $s_b$, its missing wedge is rotated too. If we compared the transformed $s_b$ with
$s_a$ directly, the network could raise the score by lining up the *empty wedges* instead of the molecules (Module 12's
trap). Gum-Net therefore **fills** the missing Fourier coefficients of the transformed $s_b$ with those of $s_a$:

$$\mathcal F(\hat s_b)(\xi)=\begin{cases}\mathcal F(T_\phi(s_b))(\xi), & m_\phi(\xi)=1 \quad\text{(measured)}\\[2pt] \mathcal F(s_a)(\xi), & m_\phi(\xi)=0 \quad\text{(missing)}\end{cases}$$

where $m_\phi$ is the wedge mask rotated by the predicted rotation (only the rotation matters, because Fourier
magnitudes ignore shifts). The training loss is then simply **one minus the Pearson correlation**:

$$\mathcal L = 1-\frac{\sum_i (s_{a,i}-\bar s_a)(\hat s_{b,i}-\bar{\hat s}_b)}{\sqrt{\sum_i (s_{a,i}-\bar s_a)^2}\sqrt{\sum_i(\hat s_{b,i}-\bar{\hat s}_b)^2}}.$$

*In words:* "after you transform $s_b$, it should look like $s_a$ wherever $s_b$ actually has data." No pose labels
appear anywhere — this is what makes the method **unsupervised**. *(Note: the paper writes the loss as $1-r$; the
released code uses $1-r^2$.)*

In [ ]:
# ── 1.4 Imputation and loss ──
def impute(sa, sb_transformed, theta):
    """Fill the (rotated) missing wedge of the transformed s_b with the Fourier coefficients of s_a."""
    m = wedge2d(theta)
    return torch.fft.ifft2(torch.fft.fft2(sb_transformed) * m + torch.fft.fft2(sa) * (1 - m)).real


def pearson_loss(a, b):
    """1 − Pearson correlation, per pair."""
    a, b = a.flatten(1), b.flatten(1)
    a, b = a - a.mean(1, keepdim=True), b - b.mean(1, keepdim=True)
    return 1 - (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-8)


print("loss with the TRUE transformation (noise-free pairs):",
      np.round(pearson_loss(sa, impute(sa, restored, back_th)).numpy(), 3))
print("loss with NO transformation                         :",
      np.round(pearson_loss(sa, impute(sa, sb, torch.zeros(4))).numpy(), 3))

🔍 The loss is near 0 for the true transformation and large without it — so lowering the loss means finding the
transformation. The network only ever sees this number.

---
## 2 · Gum-Net-lite: assemble, train without labels, evaluate

Our 2D network follows Gum-Net's layout at a much smaller size: three convolution blocks with two spectral-pooling steps
(32 → 16 → 8), unit-length feature vectors, the two-way correlation layer, one convolution per correlation map, two
fully connected layers, and a **sigmoid** output that keeps the parameters in range,
$\theta = (2p_1-1)\,\theta_{\max}$ and $t = (2p_{2,3}-1)\,t_{\max}$ (with $t_{\max}=4\sqrt2$ px). *(The 3D original has five 3D convolution layers
with 32–512 filters, works on $32^3$ subtomograms, and regresses 3 ZYZ Euler angles and 3 shifts.)*

In [ ]:
# ── 2.1 The network ──
class GumNetLite(nn.Module):
    """A 2D, miniature Gum-Net: Siamese features → two-way correlation → regression of (θ, t_x, t_y)."""

    def __init__(self, pooling="spectral"):
        super().__init__()
        pool = (lambda n, m: SpectralPool2d(n, m)) if pooling == "spectral" else (lambda n, m: nn.MaxPool2d(n // m))
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), pool(32, 16),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), pool(16, 8),
            nn.Conv2d(32, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU())
        self.head_ab = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())
        self.head_ba = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())
        self.regress = nn.Sequential(nn.Linear(2 * 32 * 64, 128), nn.ReLU(), nn.Linear(128, 3))

    def forward(self, sa, sb):
        fa = F.normalize(self.features(sa), dim=1)                 # unit-length feature vector per position
        fb = F.normalize(self.features(sb), dim=1)
        h = torch.cat([self.head_ab(correlation(fa, fb)).flatten(1),
                       self.head_ba(correlation(fb, fa)).flatten(1)], 1)
        p = torch.sigmoid(self.regress(h))                         # bounded outputs in (0, 1)
        return (2 * p[:, 0] - 1) * MAX_ANGLE, (2 * p[:, 1:] - 1) * OUT_SHIFT


def train_gumnet(model, steps, snr, lr, batch=32, log_every=100):
    """Unsupervised training: minimise 1 − r between s_a and the transformed, wedge-imputed s_b."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train(); history = []
    for step in range(steps):
        sa, sb, _, _ = make_pairs(bank, batch, snr)
        th_hat, t_hat = model(sa, sb)
        loss = pearson_loss(sa, impute(sa, transform(sb, th_hat, t_hat), th_hat.detach())).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if step % log_every == 0 or step == steps - 1:
            history.append((step, loss.item()))
    return history


def angle_errors(model, snr, n=200, seed=1):
    """Absolute error (degrees) of the predicted angle on never-seen test views."""
    model.eval()
    with torch.no_grad():
        sa, sb, th, t = make_pairs(test_bank, n, snr, generator=torch.Generator().manual_seed(seed))
        th_hat, t_hat = model(sa, sb)
        th_true, t_true = undo_params(th, t)
    return torch.rad2deg((th_hat - th_true).abs()).numpy(), (t_hat - t_true).norm(dim=1).numpy()


torch.manual_seed(SEED)
gum = GumNetLite("spectral")
print(f"Gum-Net-lite: {sum(p.numel() for p in gum.parameters()):,} parameters")
e0, _ = angle_errors(gum, 100.0)
print(f"before training: median angle error {np.median(e0):.1f}° (an untrained network outputs ~ a constant)")

### 2.2 Training — first on clean pairs, then fine-tuning on noisy ones

We follow the strategy of the lab's released Gum-Net demo, which **fine-tunes a model pre-trained at SNR 100** on noisier
data: ~700 steps on nearly noise-free pairs, then ~300 steps at SNR 1. No step ever uses the true angles.

In [ ]:
# ── 2.2 Unsupervised training ──
t0 = time.time()
hist_clean = train_gumnet(gum, steps=700, snr=100.0, lr=2e-3)
t_pre = time.time() - t0
e_clean, s_clean = angle_errors(gum, 100.0)
t0 = time.time()
hist_noisy = train_gumnet(gum, steps=300, snr=1.0, lr=5e-4)
t_fine = time.time() - t0
e_noisy, s_noisy = angle_errors(gum, 1.0)
print(f"pre-training {t_pre:.0f} s → clean test pairs: median angle error {np.median(e_clean):.1f}°, "
      f"median shift error {np.median(s_clean):.2f} px")
print(f"fine-tuning  {t_fine:.0f} s → SNR-1 test pairs: median angle error {np.median(e_noisy):.1f}°, "
      f"median shift error {np.median(s_noisy):.2f} px")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(*zip(*hist_clean), "o-", ms=3, label="pre-training (SNR 100)")
ax.plot(*zip(*[(s + 700, l) for s, l in hist_noisy]), "o-", ms=3, label="fine-tuning (SNR 1)")
ax.set_xlabel("training step"); ax.set_ylabel("loss 1 − r (batch)"); ax.legend(fontsize=8)
ax.set_title("Learning to align from the loss alone"); plt.show()

### 2.3 Comparison with exhaustive search

Our baseline is the classical approach of Module 12 in 2D: try every angle from −60° to +60° in 2° steps, find the best
shift for each with an FFT, score with **wedge-constrained** correlation, keep the best.

In [ ]:
# ── 2.3 Exhaustive search baseline ──
def exhaustive_search(sa, sb, step_deg=2.0):
    """For each pair: best angle over a grid, with FFT translation search and wedge-constrained correlation."""
    angles = torch.deg2rad(torch.arange(-60, 60 + step_deg, step_deg))
    best = []
    for i in range(len(sa)):
        A = sa[i:i + 1].repeat(len(angles), 1, 1, 1)
        Bw = transform(sb[i:i + 1].repeat(len(angles), 1, 1, 1), angles, torch.zeros(len(angles), 2))
        m = wedge2d(angles)
        Fa, Fb = torch.fft.fft2(A) * m, torch.fft.fft2(Bw) * m
        Fa[..., 0, 0] = 0; Fb[..., 0, 0] = 0                       # remove the means
        cc = torch.fft.ifft2(Fa * Fb.conj()).real.flatten(1)
        cc = cc / (Fa.abs().pow(2).sum((1, 2, 3)).sqrt() * Fb.abs().pow(2).sum((1, 2, 3)).sqrt())[:, None]
        best.append(angles[cc.max(1).values.argmax()])
    return torch.stack(best)


with torch.no_grad():
    sa_t, sb_t, th_t, t_t = make_pairs(test_bank, 200, 1.0, generator=torch.Generator().manual_seed(1))
    th_true, _ = undo_params(th_t, t_t)
    t0 = time.time(); th_search = exhaustive_search(sa_t, sb_t); t_search = time.time() - t0
    t0 = time.time(); th_net, _ = gum(sa_t, sb_t); t_net = time.time() - t0
e_search = torch.rad2deg((th_search - th_true).abs()).numpy()
print(f"exhaustive search (61 angles × all shifts): median error {np.median(e_search):.1f}°   time {1000 * t_search / 200:.1f} ms/pair")
print(f"Gum-Net-lite (one forward pass)            : median error {np.median(e_noisy):.1f}°   time {1000 * t_net / 200:.2f} ms/pair")


fig, ax = plt.subplots(figsize=(6, 3.2))
bins = np.linspace(0, 60, 25)
ax.hist(e_search, bins, alpha=0.6, label="exhaustive search (SNR 1)")
ax.hist(e_noisy, bins, alpha=0.6, label="Gum-Net-lite (SNR 1)")
ax.set_xlabel("angle error (°)"); ax.set_ylabel("test pairs"); ax.legend(fontsize=8)
ax.set_title("Angle errors on 200 never-seen test pairs"); plt.show()

🔍 **What just happened?** After about a minute of label-free training on a CPU, the tiny network aligns never-seen
views to within roughly the same error range as an exhaustive search — a few degrees worse in our toy. The errors of
*both* methods come from noise, the missing wedge and the smoothness of 1-nm projections (views a few degrees apart
look almost identical). The network's advantage is speed: one forward pass replaces a full search. In 2D the search is
cheap (61 angles), so the gain is modest; in 3D a 10° grid already has ~15,000 rotations, which is where Gum-Net's reported
**70–110× speed-up** comes from. With a full-size 3D network, much longer training and 3D data, the paper found Gum-Net
*more* accurate than the classical methods at low SNR — our miniature does not reproduce that.

✏️ **Your turn.** (1) Fine-tune at `snr=0.3` (cell 2.2) and re-run 2.3: which method degrades faster? (In the paper,
at SNR 0.01 Gum-Net's rotation error was 1.50 versus 2.38 for the classical methods, in Frobenius-norm units.)
(2) Train `GumNetLite("max")` with the same budget — the paper's max-pooling ablation. Does our 2D toy show a difference?

---
## 3 · Reference-free averaging with the trained aligner

Gum-Net's averaging needs no reference: start from the plain average of all copies, then repeat "**align every copy to
the current average, re-average**". We try it on 80 noisy (SNR 1), randomly rotated and shifted copies of one view.

In [ ]:
# ── 3.1 Iterative reference-free averaging ──
gen = torch.Generator().manual_seed(5)
truth_img = test_bank[7][None, None]
n_copies = 80
th_c = (torch.rand(n_copies, generator=gen) * 2 - 1) * MAX_ANGLE
t_c = (torch.rand(n_copies, 2, generator=gen) * 2 - 1) * MAX_SHIFT
copies = transform(truth_img.repeat(n_copies, 1, 1, 1), th_c, t_c)
copies = copies + torch.randn(copies.shape, generator=gen) / math.sqrt(1.0)
copies = torch.fft.ifft2(torch.fft.fft2(copies) * wedge2d(torch.zeros(n_copies))).real

gum.eval()
average = copies.mean(0, keepdim=True)
snapshots = [average]
with torch.no_grad():
    for it in range(4):
        th_hat, t_hat = gum(average.repeat(n_copies, 1, 1, 1), copies)
        aligned = transform(copies, th_hat, t_hat)        # no imputation here: average only measured data
        average = aligned.mean(0, keepdim=True)
        snapshots.append(average)
best_possible = transform(copies, *undo_params(th_c, t_c)).mean(0, keepdim=True)
fig, axes = plt.subplots(1, 7, figsize=(11, 1.9))
panels = [(truth_img, "true view")] + [(s, f"iteration {i}") for i, s in enumerate(snapshots)] + \
         [(best_possible, "true poses (oracle)")]
for ax, (im, lab) in zip(axes, panels):
    ax.imshow(im[0, 0]); ax.axis("off")
    r = pearson(im[0, 0].numpy(), truth_img[0, 0].numpy())
    ax.set_title(f"{lab}\nr = {r:.2f}", fontsize=8)
plt.tight_layout(); plt.show()

🔍 Iteration 0 (the unaligned average) is a blur. One round of "align to the average, re-average" sharpens it and raises
the correlation with the true view ($r$); further rounds change little. The gap to the oracle average (computed with the
true poses) is the price of our toy aligner's ~15° errors — **an average is only as good as the alignments that built
it**. Note that we did *not* impute missing-wedge data when averaging: filling every copy with the reference's
coefficients would copy the reference into the average — the reference bias of Module 14. Gum-Net used this
align-and-re-average loop for its averaging experiments (500 simulated ribosome subtomograms per SNR level, and several
real datasets), and reported averages with about 10 % better resolution than classical alignment.

---
## 4 · Equivariance, transformers and BOE-ViT

### 4.1 What equivariance means

A function $f$ is **equivariant** to a group of transformations $G$ if transforming the input transforms the output in
a predictable way:

$$f(T_g\,x) = T'_g\,f(x)\qquad\text{for all } g\in G.$$

Convolutions are (nearly) **shift-equivariant**: shift the image and the feature map shifts. That is why CNNs suit
alignment. **Vision transformers (ViTs)** cut the image into patches (e.g. 4×4 pixels), turn each patch into a token,
and process tokens with attention. Cutting into patches on a fixed grid **breaks shift equivariance**: shift the image
by 1 pixel and every patch now contains different pixels — the tokens change, not just move. Let us see it.

In [ ]:
# ── 4.1 Patch embedding is not shift-equivariant ──
P = 4                                                         # patch size
torch.manual_seed(SEED)
embed = nn.Conv2d(1, 8, kernel_size=P, stride=P, bias=False)  # standard ViT patch embedding (linear map of each patch)
x = test_bank[3][None, None]


def token_shift_error(tokens0, tokens_s, shift_px):
    """How far tokens_s is from tokens0 moved by the closest whole number of patches (relative error)."""
    k = round(shift_px / P)
    moved = torch.roll(tokens0, shifts=k, dims=-1)
    return float((tokens_s - moved).norm() / tokens0.norm())


with torch.no_grad():
    t0_ = embed(x)
    errs_plain = [token_shift_error(t0_, embed(torch.roll(x, s, dims=-1)), s) for s in range(9)]
print("relative token error after a horizontal shift of s pixels (s = 0..8):")
print("  plain patch embedding:", np.round(errs_plain, 2))

Shifts by a whole patch (s = 0, 4, 8) just move the tokens (error 0); any other shift scrambles them.

### 4.2 BOE-ViT's fix: polyphase anchoring

Split the image into its $P\times P$ **polyphase components** — the sub-images $X^{(p,q)}$ made of every $P$-th pixel
starting at offset $(p,q)$. Shifting the image by one pixel just *relabels* which component is which. BOE-ViT anchors
the patch grid on the component with the **largest norm**, $\arg\max_{(p,q)}\lVert X^{(p,q)}\rVert$, before cutting
patches. Whatever the shift, the grid then starts at the "same" pixels of the object, so a shift of the input becomes a
clean shift of the token grid (the paper's Lemma E.1: $P(T_gX)=T_{g'}(P(X))$ with $g'$ a whole number of patches).

In [ ]:
# ── 4.2 Polyphase anchoring restores shift equivariance ──
def anchored_embed(x):
    """Roll the image so that its max-norm polyphase component starts at (0, 0), then patch-embed.
    Returns the tokens and the anchor offset (which records where the grid was placed)."""
    norms = torch.stack([torch.stack([x[..., p::P, q::P].norm() for q in range(P)]) for p in range(P)])
    p, q = divmod(int(norms.argmax()), P)
    return embed(torch.roll(x, shifts=(-p, -q), dims=(-2, -1))), (p, q)


with torch.no_grad():
    a0, anchor0 = anchored_embed(x)
    errs_anch = []
    for s in range(9):
        a_s, anchor_s = anchored_embed(torch.roll(x, s, dims=-1))
        k = (s + anchor0[1] - anchor_s[1]) // P                 # whole-patch shift implied by the anchors
        errs_anch.append(float((a_s - torch.roll(a0, shifts=k, dims=-1)).norm() / a0.norm()))
print("  anchored embedding   :", np.round(errs_anch, 2))
fig, ax = plt.subplots(figsize=(5.6, 3))
ax.plot(range(9), errs_plain, "o-", label="plain patch embedding")
ax.plot(range(9), errs_anch, "s-", label="polyphase-anchored (BOE-ViT idea)")
ax.set_xlabel("input shift (pixels)"); ax.set_ylabel("token error vs. shifted tokens"); ax.legend(fontsize=8)
ax.set_title("Shift equivariance of patch embeddings"); plt.show()

🔍 With anchoring, every shift — not only whole-patch shifts — leaves the tokens unchanged except for a clean move of the
grid (error ≈ 0). *(Circular shifts make this exact here; at image borders it holds approximately.)*

### 4.3 Rotations: multi-axis rotation encoding (explained, not implemented)

For rotations BOE-ViT introduces **MARE** (multi-axis rotation encoding), a relative of the "rotary position
encodings" used in language models. For each axis $a\in\{d,h,w\}$ a learned matrix $A_a$ turns the token position $p$
into a rotation vector $w_a = A_a p$; this defines a rotation $R_a=\exp(\hat W_a)$ (with $\hat W_a$ the skew-symmetric
matrix of $w_a$) that is applied to the attention queries and keys before computing attention,

$$Q'_a = QR_a^{\mathsf T},\quad K'_a=KR_a^{\mathsf T},\qquad \mathrm{Attn}_a=\mathrm{softmax}\!\Big(\tfrac{Q'_aK'^{\mathsf T}_a}{\sqrt{d_k}}\Big)V,\qquad \mathrm{Attn}_{\text{MARE}}=\sum_a\mathrm{Attn}_a .$$

*In words:* the similarity between two tokens is computed after rotating their descriptions according to *where* they
are, so that a global rotation of the input produces a corresponding rotation of the attention output (the paper's
Lemma E.3) instead of an arbitrary change. Together with polyphase anchoring, this makes the transformer respect the
geometry of the alignment problem.

---
## 5 · Self-supervised pose regression (BOE-ViT's training signal, in miniature)

BOE-ViT is called *self-supervised* because its training labels are **transformations the authors applied themselves**
to simulated subtomograms: take $x$, apply a known random $(\theta, t)$, and train the network to predict them with

$$\mathcal L = \alpha\,\mathrm{MSE}(\theta,\hat\theta)+\beta\,\mathrm{MSE}(t,\hat t)\qquad(\alpha=1,\ \beta=2\ \text{worked best}).$$

Unlike Gum-Net's loss, this uses the known transformations — available for free in simulation. We reuse Gum-Net-lite's
architecture with this loss (predicting the angle through its sine and cosine is not needed in our ±45° range).

In [ ]:
# ── 5.1 Training on self-generated transformations ──
torch.manual_seed(SEED)
reg = GumNetLite("spectral")
opt = torch.optim.Adam(reg.parameters(), lr=2e-3)
alpha, beta = 1.0, 2.0
t0 = time.time()
reg.train()
for step in range(500):
    sa, sb, th, t = make_pairs(bank, 32, snr=100.0 if step < 350 else 1.0)
    th_true, t_true = undo_params(th, t)
    th_hat, t_hat = reg(sa, sb)
    loss = alpha * F.mse_loss(th_hat, th_true) + beta * F.mse_loss(t_hat / MAX_SHIFT, t_true / MAX_SHIFT)
    opt.zero_grad(); loss.backward(); opt.step()
e_reg, s_reg = angle_errors(reg, 1.0)
print(f"self-supervised regression ({time.time() - t0:.0f} s, 500 steps): median angle error {np.median(e_reg):.1f}°, "
      f"shift error {np.median(s_reg):.2f} px at SNR 1")
print(f"for comparison — unsupervised Gum-Net-lite: {np.median(e_noisy):.1f}°, exhaustive search: {np.median(e_search):.1f}°")

🔍 With labels that cost nothing to make, the same network learns pose regression directly — in our toy even more
accurately than with the unsupervised loss, because every training step is told the exact answer. The catch is the
**simulation-to-reality gap** (Module 20): labels exist only for simulated data, while Gum-Net's unsupervised loss can
be trained on real subtomograms. BOE-ViT's experiments were on simulated data (five complexes, SNR 0.1–0.01), where its
equivariant design cut rotation error by 77.3 % and translation error by 62.5 % relative to the previous best methods
at SNR 0.01.

✏️ **Your turn.** Train the regression network only on clean pairs (`snr=100.0` for all 500 steps) and evaluate at SNR 1.
What does that tell you about matching training and test conditions?

---
### 📄 Paper connection — Gum-Net (Zeng & Xu, CVPR 2020)

**Zeng X, Xu M.** *Gum-Net: Unsupervised geometric matching for fast and accurate 3D subtomogram image alignment and
averaging.* Proc. IEEE/CVF CVPR 2020, pp. 4073–4084.
[CVF open access](https://openaccess.thecvf.com/content_CVPR_2020/html/Zeng_Gum-Net_Unsupervised_Geometric_Matching_for_Fast_and_Accurate_3D_Subtomogram_CVPR_2020_paper.html).
- **Question:** align two noisy, missing-wedge subtomograms fast, without ground-truth transformations for training.
- **What the paper does:** a Siamese 3D CNN with **DCT spectral pooling and filtering**; a **two-way feature-correlation
  layer**; regression of 6 parameters (ZYZ Euler angles and shifts) through a sigmoid (last layer initialised with zero
  weights and a random bias); a differentiable **3D spatial transformer**; **missing-wedge imputation** of the
  transformed subtomogram from its partner; and the unsupervised loss $1-r$ (code: $1-r^2$). Averaging is reference-free
  by iterating alignment to the running average.
- **What our toy keeps:** every component, in 2D (3 parameters), on projections of code-generated zoo molecules; pre-training on
  clean data then fine-tuning on noisy data, as in the released demo.
- **What it leaves out:** 3D, the network depth and widths ($32^3$ input, 32–512 filters), the 5,000-random-pairs-per-epoch
  training regime, the simulated benchmark (5 complexes, SNR 100 to 0.01, 30° wedge) and the real datasets.
- **Result reported:** alignment error reduced by 40–50 %, averaging resolution improved by ~10 %, and a 70–110× speed-up
  (17.6 s per 1,000 subtomograms on one GPU versus 1,916 s and 1,251 s for two classical methods on one CPU core).
  Rotation/translation errors at SNR 0.01: Gum-Net 1.50 / 6.78 voxels versus 2.38 / 11.3 for the classical methods
  (rotation error measured as $\lVert R_{est}-R_{gt}\rVert_F$). Gum-Net was not best everywhere (e.g. rotation at SNR 100).
- **Code:** [AITom `align/deep/gum`](https://github.com/xulabs/aitom/tree/master/aitom/align/deep/gum) (Keras/TensorFlow).

### 📄 Paper connection — BOE-ViT (Jiang *et al.*, CVPR 2025)

**Jiang R, Daggett J, Pingulkar S, Zhao Y, Dhingra P, Brown D, Wu Q, Zeng X, Li X, Xu M.** *BOE-ViT: Boosting
orientation estimation with equivariance in self-supervised 3D subtomogram alignment.* Proc. IEEE/CVF CVPR 2025,
pp. 29352–29362.
[CVF open access](https://openaccess.thecvf.com/content/CVPR2025/html/Jiang_BOE-ViT_Boosting_Orientation_Estimation_with_Equivariance_in_Self-Supervised_3D_Subtomogram_CVPR_2025_paper.html).
- **Question:** make vision transformers accurate for subtomogram pose estimation at very low SNR by building in
  equivariance to shifts and rotations.
- **What the paper does:** **Polyshift** (polyphase-anchored, shift-equivariant patch embedding, with a group-equivariant
  CNN on the polyphase components, circular-padded convolutional position encoding and a feature comparator between the
  input and target embeddings) and **MARE** (multi-axis rotation encoding of queries and keys); trained on self-generated
  transformations of simulated subtomograms with $\alpha\,\mathrm{MSE}(\theta)+\beta\,\mathrm{MSE}(t)$; 4 transformer
  blocks, 4 heads, hidden size 60, patch $4^3$.
- **What our toy keeps:** the shift-equivariance failure of patch embedding and its polyphase fix (exactly, in 2D), and the
  self-supervised parameter-regression loss.
- **What it leaves out:** the transformer itself, MARE (explained only), the group-equivariant CNN, 3D.
- **Result reported:** on simulated data (complexes 1I6V, 6A5L, 5LQW, 5T2C, 5MPA; test SNR 0.1–0.01) the abstract
  reports 77.3 % lower rotation error and 62.5 % lower translation error than the previous state of the art at SNR 0.01.
  ⚠️ We could read the abstract and supplement but not the main text; the units of the rotation errors in the supplement
  tables (degrees or radians) should be checked in the paper. No real-data experiments are described in the supplement.

## ✅ Checkpoint

1. Why is Gum-Net called *unsupervised* even though it is trained with a loss?
   <details><summary>Answer</summary>The loss (1 − correlation between s_a and the transformed s_b) needs no true poses or labels — only the two subtomograms.</details>
2. What would go wrong without missing-wedge imputation?
   <details><summary>Answer</summary>The network could increase the correlation by lining up the empty wedges (the missing-wedge artefacts) rather than the molecules.</details>
3. Why spectral pooling instead of max pooling?
   <details><summary>Answer</summary>Max pooling discards the position of a feature within each block; spectral pooling (keeping low DCT frequencies) downsamples while preserving sub-pixel position information, which alignment needs.</details>
4. What does the correlation layer output for 8×8 feature maps, and why both directions?
   <details><summary>Answer</summary>64 similarity maps of size 8×8 (every position of one map against every position of the other); computing a→b and b→a gives the regressor symmetric evidence.</details>
5. Define shift equivariance and explain how polyphase anchoring achieves it for patch embeddings.
   <details><summary>Answer</summary>f(shifted x) = shifted f(x). Anchoring places the patch grid on the max-norm polyphase component, so the grid moves with the object and an input shift becomes a whole-token shift.</details>
6. What is "self-supervised" about BOE-ViT's training?
   <details><summary>Answer</summary>The labels are random transformations generated by the authors themselves on simulated subtomograms; the network regresses them with an MSE loss.</details>

## 🧠 Questions & discussion

1. Gum-Net is trained on pairs "regardless of their structural class". What does the network learn when $s_a$ is a
   ribosome and $s_b$ a proteasome? Is that harmful?
2. A network's answer is instant but can fail silently. How would you detect a bad alignment in practice?
3. Our errors plateau at several degrees for both the network and the search. What limits them, and how would 3D data
   change that?
4. Equivariance is a property you can *test*. Design a numerical test for rotation equivariance of a 2D CNN.
5. When would you prefer an unsupervised loss (Gum-Net) over self-generated labels (BOE-ViT), and vice versa?

## 🚀 Bonus — going further

- **Full range of angles.** Allow θ ∈ [−180°, 180°) and predict $(\cos\theta,\sin\theta)$ instead of θ. Why does a sigmoid
  output struggle with wrap-around?
- **3D.** Replace the 2D layers with `Conv3d`, `affine_grid` in 3D (see Module 13's spatial transformer) and 3D DCT
  pooling, and align $24^3$ subtomograms. Watch the training time.
- **Official code.** Run AITom's Gum-Net demo (`aitom/align/deep/gum/train_demo.py`) and compare its architecture with ours.
- **Read** Jaderberg *et al.* (spatial transformer networks) and Rocco *et al.* (geometric matching networks), the two
  ideas Gum-Net extends to 3D.

## 📝 Lab note

```text
Module 21 — date:
Input: pairs of 2D molecule views (s_a complete; s_b transformed, noisy, missing wedge)
Output: predicted (θ, t); angle errors; reference-free averages; equivariance errors
Method: Gum-Net-lite (spectral pooling, two-way correlation, spatial transformer, imputation, 1 − r);
        exhaustive search baseline; polyphase-anchored patch embedding; self-supervised regression
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Zeng & Xu 2020 (Gum-Net); Jiang et al. 2025 (BOE-ViT)
```

## 📚 References

1. Zeng X, Xu M. Gum-Net: Unsupervised geometric matching for fast and accurate 3D subtomogram image alignment and averaging. *CVPR* 2020, pp. 4073–4084.
2. Jiang R, Daggett J, Pingulkar S, Zhao Y, Dhingra P, Brown D, Wu Q, Zeng X, Li X, Xu M. BOE-ViT: Boosting orientation estimation with equivariance in self-supervised 3D subtomogram alignment. *CVPR* 2025, pp. 29352–29362.
3. Jaderberg M, Simonyan K, Zisserman A, Kavukcuoglu K. Spatial transformer networks. *NeurIPS* 2015.
4. Rocco I, Arandjelović R, Sivic J. Convolutional neural network architecture for geometric matching. *CVPR* 2017.
5. Rippel O, Snoek J, Adams RP. Spectral representations for convolutional neural networks. *NeurIPS* 2015.
6. Xu M, Beck M, Alber F. High-throughput subtomogram alignment and classification by Fourier space constrained fast volumetric matching. *J Struct Biol* 178:152–164 (2012). doi:10.1016/j.jsb.2012.02.014

## ➡️ Next

**[22 · Sorting a cell's particles without labels: Jim-Net and DISCA](22_unsupervised_clustering_disca.ipynb)** — combine
alignment with clustering to discover which kinds of molecules a tomogram contains. Course overview:
[`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).